## 1. 데이터 불러오기
> - 데이터는 Pandas 라이브러리의 pd.read_csv 함수를 활용하여 불러온다.

In [ ]:
# 상황 정의
# 
# 은닉된 상태 -> 할인수준 -> 할인율
# 보이는 관측치 -> 판매수량 -> 컬럼 O

# 2,3개 판매/주문한 경우에 할인수준 / 낮기, 높기




In [ ]:
#필요 라이브러리 
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

#한글 폰트 지정
import matplotlib
import matplotlib.font_manager as fm

fm.get_fontconfig_fonts()
font_location = 'C:/Windows/Fonts/malgun.ttf' # For Windows
font_name = fm.FontProperties(fname=font_location).get_name()
matplotlib.rc('font', family=font_name)

In [ ]:
#데이터 경로 지정
order_data_file_path = "./data/order_detail.csv"
customer_data_file_path = "./data/customer_data.csv"

#데이터 불러오기
order_data = pd.read_csv(order_data_file_path)
customer_data = pd.read_csv(customer_data_file_path)

In [ ]:
#주문일자 컬럼을 생성한다.
order_data["주문일자"] = order_data["주문일시"].str.slice(start=0, stop=10)
order_data["주문일자"] = pd.to_datetime(order_data["주문일자"], format="%Y-%m-%d")

## 2. 마르코프 모형을 지정하기 위한 라벨링

In [ ]:
order_data

In [ ]:
order_data["주문수량"] = order_data["주문수량"].astype(str)

In [ ]:
# 1만원 / 8천원 -> 1 - (8000/10000) = 0.2 -> 20%
order_data["할인율"] = 1 - (order_data["판매가"]/order_data["정상가"])

In [ ]:
order_data.loc[(order_data["할인율"]>=0)&(order_data["할인율"]<0.3),"할인구분"] = "NORMAL"
order_data.loc[(order_data["할인율"]>=0.3)&(order_data["할인율"]<0.4),"할인구분"] = "SEASON OFF"
order_data.loc[(order_data["할인율"]>=0.4),"할인구분"] = "OUTLET"

In [ ]:
order_data = order_data.dropna()

In [ ]:
order_data

## 3. 은닉 마르코프 모형 시퀀스 만들기

In [ ]:
#고객별로 주문한 히스토리 리스트 
#홍길동 -> NORMAL, NORMAL, OUTLET (히든 시퀀스)
#       -> 1, 1, 3 (관측 시퀀스)

In [ ]:
order_data.groupby('고객번호')

In [ ]:
customer_data_sequence = []
for customerID, group in order_data.groupby('고객번호'):
    sales_level_sequence = group['할인구분'].tolist()
    purchase_amount_sequence = group['주문수량'].tolist()
    customer_data_sequence.append((customerID, sales_level_sequence, purchase_amount_sequence))

In [ ]:
customer_data_sequence

In [ ]:
len(customer_data_sequence)

## 4. 은닉 마르코프 모형 실행하기

In [ ]:
import numpy as np


# Hidden states
hidden_states = ["NORMAL","SEASON OFF","OUTLET"]

# Observed states
observed_states = ["1", "2", "3"]

In [ ]:
# 상태 전이 행렬 Transition probability 함수 만들기
def calculate_transition_matrix(customer_data, hidden_states):
    transition_counts = {state: {next_state: 0 for next_state in hidden_states} \
                         for state in hidden_states}
    # transition_counts : 상태 간 전이 횟수를 저장하는 딕셔너리
    # transition_counts[state][next_state]는 state에서 next_state로 전이한 횟수
    # ex. transition_counts[맑음][흐림] -> 맑음에서 흐림으로 진행된 횟수
    
    for _, hidden_states_list, _ in customer_data: 
        # (고객번호, 날씨 시퀀스, 기분 시퀀스)
        for i in range(len(hidden_states_list) - 1): 
            current_state = hidden_states_list[i] 
            next_state = hidden_states_list[i + 1]
            transition_counts[current_state][next_state] += 1 
            # 상태 전이 딕셔너리의 카운트 값에서 전이 카운트를 1씩 증가
    # 맑음 - 맑음 -> 20번 40%
    # 맑음 - 흐림 -> 30번 60%
    # 흐림 - 맑음 -> 10번 20%
    # 흐림 - 흐림 -> 40번 80%
    # 100번
    transition_matrix = {state: {next_state: count \
                                 / sum(transitions.values()) \
                                 for next_state, count in transitions.items()} \
                         for state, transitions in transition_counts.items()}
    # 위에서 계산한 상태 간 전이 횟수를 저장한 딕셔너리 
    # transition_counts를 기반으로 Transition matrix 계산
    # 각 상태(state)에서 다른 상태(next_state)로 전이될 확률은 
    # 그 전이 횟수(count)를 해당 상태에서 일어난 
    # 총 전이 횟수(sum(transitions.values()))로 나눈 값입니다.
    # 최종적으로 transition_matrix는 상태 간 전이 확률을 저장하는 딕셔너리로, 
    # transition_matrix[state][next_state]는 state에서 
    # next_state로 전이될 확률을 나타냅니다.
    return transition_matrix

In [ ]:
# 출력 행렬 emission(output) probability 함수 만들기
def calculate_emission_matrix(customer_data, hidden_states, observed_states):
    emission_counts = {state: {obs_state: 0 for obs_state in observed_states} \
                       for state in hidden_states}
    # emission_counts는 각 감추어진 상태에서 
    # 각 관측된 상태로의 발생 빈도를 저장하는 딕셔너리
    # 각 감추어진 상태(state)에 대해 관측된 
    # 상태(obs_state)가 발생한 횟수를 0으로 초기화합니다.
    # 예: emission_counts[hidden_state][observed_state]는 
    # 감추어진 상태(날씨)에서 특정 관측 상태(기분)가 발생한 횟수를 나타냅니다.
    
    for _, hidden_states_list, observed_states_list in customer_data:  
        # (고객번호, 날씨 시퀀스, 기분 시퀀스)
        # customer_data에서 고객별로 상태 목록을 순회합니다. 
        # 여기서 튜플의 첫 번째 값은 무시, 
        # observed_states_list(관측된 상태 목록)와 
        # hidden_states_list(감추어진 상태 목록)를 가져옵니다.
        for obs_state, hidden_state in zip(observed_states_list, \
                                           hidden_states_list):
            #관측된 상태 목록과 감추어진 상태 목록을 동시에 순회합니다. 
            #이때, zip을 사용해 각 고객의 
            # 관측된 상태(obs_state)와 감추어진 상태(hidden_state)를 대응시킵니다.
            emission_counts[hidden_state][obs_state] += 1
            # 감추어진 상태(hidden_state)에서 
            # 관측된 상태(obs_state)가 발생한 횟수를 1씩 증가시킵니다. 
            # 즉, emission_counts[hidden_state][obs_state]는 
            # 해당 감추어진 상태에서 해당 관측된 상태가 나타난 빈도를 기록합니다.

    emission_matrix = {state: {obs_state: count / sum(emissions.values()) \
                               for obs_state, count in emissions.items()} \
                       for state, emissions in emission_counts.items()}
    # emission_counts에서 발생 횟수를 기반으로 확률을 계산하여 
    # Emission Matrix를 만듭니다.
    # 각 감추어진 상태(state)에서 특정 관측 상태(obs_state)가 발생할 확률은 
    # 그 발생 횟수(count)를 해당 감추어진 상태에서 
    # 발생한 모든 관측 상태의 총합(sum(emissions.values()))으로 나눈 값입니다.
    return emission_matrix

    # 최종적으로 emission_matrix는 감추어진 상태에서 
    # 각 관측 상태로의 확률을 저장하는 딕셔너리입니다. 
    # 예를 들어, emission_matrix[hidden_state][observed_state]는 
    # 감추어진 상태에서 관측 상태가 발생할 확률을 나타냅니다.

In [ ]:
# transition 계산하기
transition_matrix = calculate_transition_matrix(customer_data_sequence, \
                                                hidden_states)
print("Transition 결과:")
for state, transitions in transition_matrix.items():
    print(state + ": ", transitions)

# emission 계산하기
emission_matrix = calculate_emission_matrix(customer_data_sequence, \
                                            hidden_states, observed_states)
print("\nEmission(output) 결과:")
for state, emissions in emission_matrix.items():
    print(state + ": ", emissions)